# Automated Corporate Brochure Generator

## Problem Statement

### 📌 Context
Modern businesses, investors, and sales teams constantly need to evaluate new companies, clients, and partners. Generating personalized informational material (like brochures, summaries, or prospectuses) for these entities typically requires hours of manual research, website reading, and content writing.

### 🛑 The Challenge
The goal is to build an automated product that dynamically generates a comprehensive corporate brochure for any given company. The tool must accept only two inputs:
1. **Company Name**
2. **Primary Website URL**

### 💡 Objective
Create an end-to-end AI pipeline that automates this workflow by:
* **Web Scraping:** Programmatically crawling and extracting core content, links, and details from the provided company website.
* **LLM Processing:** Utilizing Large Language Models (LLMs) to synthesize raw scraped web data into highly structured, engaging professional copy.
* **Asset Generation:** Producing a clean informational brochure targeted at prospective clients, active investors, and potential recruits.


In [2]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
from scraper import fetch_website_contents, fetch_website_links
load_dotenv(override = True)


True

In [6]:
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url= gemini_base_url, api_key=gemini_api_key)

In [18]:
link_system_prompt = """
You are an expert data extraction assistant. Your job is to filter a list of webpage links and select only the most relevant ones needed to build a comprehensive corporate company brochure.

Target Page Types to Keep:
- Corporate Identity: About Us, Company Profile, History, Mission/Vision.
- Employment: Careers, Jobs, Work with Us, Teams.
- Offerings: Products, Services, Solutions, Features.
- Contact: Contact Us, Support, Locations, Offices.

Pages to Strictly Exclude:
- Legal & Compliance: Terms of Service, Privacy Policy, Cookie Policy, End User License Agreements.
- Communications: Raw email links (mailto:), social media sharing buttons, or developer documentation APIs.

CRITICAL OUTPUT REQUIREMENTS:
1. You must respond ONLY with a raw, valid JSON object matching the exact schema below.
2. Do not wrap the response in markdown blocks (no ```json or ```).
3. Do not include any conversational filler, explanations, or text before/after the JSON.
4. If no relevant links are found, return an empty list for the "links" key.

JSON Schema Response Format:
{
    "links": [
        {
            "type": "Clear category name (e.g., 'about page', 'careers page', 'products page')",
            "url": "The full absolute URL string"
        }
    ]
}
"""


In [19]:
def get_user_link_prompt(url):
    user_prompt= f"""Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):
"""
    links = fetch_website_links(url)
    user_prompt+= "\n".join(links)
    return user_prompt

In [23]:
def select_relevent_links(url):
    print(f"Selecting relevant links for {url} by calling Gemini-2.5-flash")
    user_prompt = get_user_link_prompt(url) 
    responses = gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": link_system_prompt}, 
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    
   
    result = responses.choices[0].message.content
    

    links_dict = json.loads(result)
    
   
    actual_links_list = links_dict.get('links', [])
    print(f"Found {len(actual_links_list)} relevant brochure links out of the total crawled.")
    
    return links_dict


In [ ]:
select_relevent_links("https://www.cloudsek.com/")

In [25]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevent_links(url)
    result = f"## Landing page:\n\n{contents}\n ## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://www.cloudsek.com/"))

In [27]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    return user_prompt

In [ ]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")